# 03 — Model eğitimi ve karşılaştırma

**Amaç:** Ön işlenmiş train/test verisiyle birden fazla sınıflandırıcı eğitmek, **test** setinde metrikleri karşılaştırmak ve en uygun modeli `models/best_model.pkl` olarak kaydetmek.

**Ön koşul:** `02_preprocessing.ipynb` çalıştırılmış olmalı; `data/processed/train_test_split.pkl` içinde `X_train`, `X_test`, `y_train`, `y_test`, ölçekleyici meta verisi bulunur.

**Notlar:**
- Değerlendirme **test** kümesi üzerinde yapılır (eğitim setiyle övünülmez).
- Sınıflar dengesiz olduğu için `accuracy` tek başına yeterli değildir; **recall**, **F1**, **ROC-AUC** birlikte okunur.
- Bu notebookta ham CSV tekrar işlenmez; sütun sırası `bundle["feature_columns"]` ile sabittir.

Her kod hücresinden önce kısa markdown açıklaması gelir; kod içinde `#` yorumları satır mantığını özetler.

### Veri paketini yükleme (bir sonraki hücre)

`joblib` ile kaydedilmiş sözlük okunur. Jupyter çalışma dizini **proje kökü** veya **`notebooks/`** olabileceği için `data/processed/...` ve `../data/processed/...` yolları sırayla denenir. Çıktı: eğitim/test matrisleri ve hedef vektörleri (`y` içinde 0 = kalan, 1 = churn).

In [1]:
# --- Ön işlenmiş veriyi yükle ---
# Dosya yolu ve serileştirilmiş nesneler
from pathlib import Path

import joblib

import pandas as pd


def _processed_pkl() -> Path:
    """İşlenmiş paket dosyasının mutlak yolunu bul."""
    # Çalışma dizinine göre iki olası göreli yol dene
    for rel in (
        Path("data/processed/train_test_split.pkl"),
        Path("../data/processed/train_test_split.pkl"),
    ):
        r = rel.resolve()
        if r.is_file():
            return r
    raise FileNotFoundError("train_test_split.pkl yok. Önce 02_preprocessing.ipynb çalıştır.")


p = _processed_pkl()

# Ön işlemede kaydedilen sözlüğü belleğe al (X, y, scaler meta, feature_columns, ...)
bundle = joblib.load(p)
# Özellik matrisleri (satır = müşteri, sütun = 30 kodlanmış özellik)
X_train = bundle["X_train"]
X_test = bundle["X_test"]
# Hedef: 0 = churn yok, 1 = churn
y_train = bundle["y_train"]
y_test = bundle["y_test"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
# Özellik isimleri listesi uzunluğu = sütun sayısı (API/tahmin ile aynı sıra korunmalı)
print("Özellik sayısı:", len(bundle["feature_columns"]))


X_train: (5634, 30) | X_test: (1409, 30)
Özellik sayısı: 30


### Kütüphaneler ve veri tipi (bir sonraki hücre)

**Metrikler:** doğruluk, kesinlik, duyarlılık (recall), F1, ROC-AUC — churn gibi dengesiz problemlerde recall ve F1 özellikle anlamlıdır.

**Modeller:** Logistic Regression, Random Forest, Gradient Boosting (sklearn), XGBoost (`scale_pos_weight`), LightGBM (`class_weight='balanced'`).

**Float dönüşümü:** `get_dummies` sonrası kalan `bool` sütunları `float64` yapmak, tüm kütüphanelerde tutarlı davranış sağlar.

In [2]:
# --- Kütüphaneler ve veri hazırlığı ---
import numpy as np
import pandas as pd

# Sınıflandırma performans ölçütleri (test seti için kullanılacak)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
import json

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Tüm modellerde aynı tohum: tekrarlanabilirlik
RANDOM_STATE = 42

# Ağaç / XGB için tutarlılık: bool sütunları float (0.0 / 1.0)
X_train_f = X_train.astype(np.float64)
X_test_f = X_test.astype(np.float64)

# XGBoost: pozitif sınıf (churn) seyrek → ağırlık = negatif sayısı / pozitif sayısı
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / max(n_pos, 1)
print("Train churn (1):", n_pos, "| scale_pos_weight:", round(scale_pos_weight, 3))

Train churn (1): 1495 | scale_pos_weight: 2.769


### Modelleri eğitmek ve metrik tablosu (bir sonraki hücre)

`evaluate_model`: verilen modeli train’de **fit** eder, test’te tahmin ve olasılık üretir, metrikleri sözlükte toplar. **Beş** estimator (Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM) eğitilir; sonuçlar `metrics_df` tablosunda toplanır. LR ve RF’de `class_weight="balanced"`; XGB’de `scale_pos_weight`; LightGBM’de `class_weight="balanced"`.

In [3]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Modeli eğit, test setinde tahmin ve olasılık üret, metrikleri paketle."""
    # Eğitim verisiyle parametreleri öğren
    model.fit(X_tr, y_tr)
    # Sınıf etiketi tahmini (0 veya 1)
    y_pred = model.predict(X_te)
    # Churn olasılığı P(y=1): ROC-AUC için gerekli
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_te)[:, 1]
    else:
        # Bazı modellerde (linear SVM vb.) sadece decision_function olur
        y_proba = model.decision_function(X_te)
    return {
        "model": name,
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, zero_division=0),
        "recall": recall_score(y_te, y_pred, zero_division=0),
        "f1": f1_score(y_te, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, y_proba),
        "estimator": model,  # sonradan kayıt ve önem grafikleri için
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


# Karşılaştırılacak modeller ve hiperparametreler (challenge’da çoklu model beklentisi)
models_to_try = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",  # sınıf frekansına göre ağırlık
        random_state=RANDOM_STATE,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,  # tüm CPU çekirdekleri
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="logloss",
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
}

results = []
fitted = {}
for name, clf in models_to_try.items():
    out = evaluate_model(name, clf, X_train_f, X_test_f, y_train, y_test)
    fitted[name] = out  # tam çıktı (tahminler + model nesnesi)
    # Tablo için sadece skaler metrikler
    results.append({k: out[k] for k in ("model", "accuracy", "precision", "recall", "f1", "roc_auc")})

# Satır = model adı; sütun = test metrikleri
metrics_df = pd.DataFrame(results).set_index("model").round(4)
metrics_df

,accuracy,precision,recall,f1,roc_auc
model,,,,,
LogisticRegression,0.7381,0.5043,0.7834,0.6136,0.8417
RandomForest,0.7743,0.5636,0.6631,0.6093,0.8390
GradientBoosting,0.7984,0.6471,0.5294,0.5824,0.8419
XGBoost,0.7608,0.5345,0.7674,0.6301,0.8367
LightGBM,0.7502,0.5199,0.7674,0.6199,0.8344


### En iyi modeli seçmek ve hata analizi (bir sonraki hücre)

Şu an **F1 skoru** en yüksek model “en iyi” sayılır (precision–recall dengesi). İş ihtiyacına göre **recall**’ı maksimize etmek istiyorsan `metrics_df["recall"].idxmax()` kullanılabilir. **Confusion matrix** hangi hataların (yanlış churn / kaçırılan churn) yapıldığını gösterir; **classification report** sınıf bazlı precision/recall/F1 verir.

In [4]:
# En iyi model: F1 ile seçim (churn için precision–recall dengesi)
# Alternatif: metrics_df["recall"].idxmax() veya metrics_df["roc_auc"].idxmax()
best_name = metrics_df["f1"].idxmax()
print("Seçilen (F1):", best_name)
print(metrics_df.loc[best_name])

# Bu modele ait tam evaluate_model çıktısı
best = fitted[best_name]
print("\n=== Confusion matrix (test) ===")
# Satırlar: gerçek; sütunlar: tahmin — TN, FP / FN, TP
print(confusion_matrix(y_test, best["y_pred"]))
print("\n=== Classification report ===")
print(
    classification_report(
        y_test, best["y_pred"], target_names=["Kalıyor (0)", "Churn (1)"]
    )
)

Seçilen (F1): XGBoost
accuracy     0.7608
precision    0.5345
recall       0.7674
f1           0.6301
roc_auc      0.8367
Name: XGBoost, dtype: float64

=== Confusion matrix (test) ===
[[785 250]
 [ 87 287]]

=== Classification report ===
              precision    recall  f1-score   support

 Kalıyor (0)       0.90      0.76      0.82      1035
   Churn (1)       0.53      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



### Özellik önemi (bir sonraki hücre)

Ağaç tabanlı modeller (`RandomForest`, `GradientBoosting`, `XGBoost`, `LightGBM`) `feature_importances_` üretir. **Logistic Regression** seçildiyse bu öznitelik yoktur; uyarı basılır.

In [5]:
# --- Özellik önemi (ağaç tabanlı modellerde) ---
est = best["estimator"]
# Ön işlemedeki sütun sırası ile aynı olmalı
feat_names = bundle["feature_columns"]

if hasattr(est, "feature_importances_"):
    # Önem vektörünü isimlendir, büyükten küçüğe sırala
    imp = pd.Series(est.feature_importances_, index=feat_names).sort_values(
        ascending=False
    )
    display(imp.head(15))
else:
    print("Bu modelde feature_importances_ yok (ör. Logistic Regression).")

Contract_Two year                     0.356359
InternetService_Fiber optic           0.124363
Contract_One year                     0.123920
OnlineSecurity_No internet service    0.065245
InternetService_No                    0.052815
StreamingMovies_Yes                   0.042489
PaymentMethod_Electronic check        0.019299
tenure                                0.019035
PhoneService                          0.015366
OnlineSecurity_Yes                    0.014938
StreamingTV_Yes                       0.014389
TechSupport_Yes                       0.013715
MultipleLines_No phone service        0.013702
PaperlessBilling                      0.013451
MultipleLines_Yes                     0.013324
dtype: float32

### Tüm modelleri dosyaya yazmak ve API için kayıt (bir sonraki hücre)

- **`models/saved/`** — Her model ayrı dosya: `LogisticRegression.joblib`, `RandomForest.joblib`, `GradientBoosting.joblib`, `XGBoost.joblib`, `LightGBM.joblib` (isimler sözlük anahtarlarıyla aynı).
- **`models/model_registry.json`** — Hangi dosyanın **en iyi** olduğu (`best_model_key`, `selection_metric`), tüm modellerin test metrikleri ve göreli dosya yolları. **API** önce bunu okuyup en iyi `.joblib` dosyasını yükler.
- **`models/best_model.pkl`** — Geriye dönük uyumluluk: en iyi estimator + meta (tek `joblib.load` ile tahmin).

In [6]:
# --- Diske kaydet: her model ayrı dosya + registry + best paketi ---
out_dir = Path("models")
if not out_dir.is_dir():
    out_dir = Path("../models")
out_dir.mkdir(parents=True, exist_ok=True)

saved_dir = out_dir / "saved"
saved_dir.mkdir(parents=True, exist_ok=True)

# Her eğitilmiş estimator'ı belirgin dosya adıyla yaz (API/production ayrıştırması kolay)
saved_files = {}
for name, out in fitted.items():
    fname = f"{name}.joblib"
    fpath = saved_dir / fname
    joblib.dump(out["estimator"], fpath)
    saved_files[name] = str(fpath.resolve())

# JSON: API buradan en iyi dosyayı seçer (göreli yollar repo köküne göre taşınabilir)
registry = {
    "version": 1,
    "selection_metric": "f1",
    "best_model_key": best_name,
    "best_model_file": f"saved/{best_name}.joblib",
    "feature_columns": feat_names,
    "random_state": RANDOM_STATE,
    "models": {},
}
for name in fitted:
    registry["models"][name] = {
        "file": f"saved/{name}.joblib",
        "metrics_test": metrics_df.loc[name].to_dict(),
    }

registry_path = out_dir / "model_registry.json"
registry_path.write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding="utf-8")
print("Registry:", registry_path.resolve())
for name, p in saved_files.items():
    print(" ", name, "->", p)

# Geriye dönük: tek dosyada en iyi model paketi
payload = {
    "model": best["estimator"],
    "model_name": best_name,
    "metrics_test": metrics_df.loc[best_name].to_dict(),
    "metrics_all": metrics_df,
    "feature_columns": feat_names,
    "random_state": RANDOM_STATE,
    "registry_file": "model_registry.json",
}
joblib.dump(payload, out_dir / "best_model.pkl")
print("best_model.pkl:", (out_dir / "best_model.pkl").resolve())

Registry: C:\Users\Can\Desktop\data\churn-project-yzta\models\model_registry.json
  LogisticRegression -> C:\Users\Can\Desktop\data\churn-project-yzta\models\saved\LogisticRegression.joblib
  RandomForest -> C:\Users\Can\Desktop\data\churn-project-yzta\models\saved\RandomForest.joblib
  GradientBoosting -> C:\Users\Can\Desktop\data\churn-project-yzta\models\saved\GradientBoosting.joblib
  XGBoost -> C:\Users\Can\Desktop\data\churn-project-yzta\models\saved\XGBoost.joblib
  LightGBM -> C:\Users\Can\Desktop\data\churn-project-yzta\models\saved\LightGBM.joblib
best_model.pkl: C:\Users\Can\Desktop\data\churn-project-yzta\models\best_model.pkl
